In [1]:
from os import PathLike
from hdfs import InsecureClient
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.functions import col
from pyspark.sql.types import StringType
from delta import *
from pyspark.sql.types import LongType, StringType, StructField, StructType, BooleanType, ArrayType, IntegerType, DoubleType

In [2]:
# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .appName("Python Spark SQL Hive integration for the EDSTD project") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:1.0.0") \
    .enableHiveSupport() \

spark = spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
spark.sql(
    """
    SHOW TABLES FROM projeto
    """
).show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|  projeto|     amazon_books|      false|
|  projeto|   amazon_credits|      false|
|  projeto|    amazon_titles|      false|
|  projeto| book_movie_adapt|      false|
|  projeto|    book_to_movie|      false|
|  projeto|books_movie_adapt|      false|
|  projeto|  netflix_credits|      false|
|  projeto|   netflix_titles|      false|
|  projeto| streaming_titles|      false|
+---------+-----------------+-----------+



In [4]:
delta_lake_path = "hdfs://hdfs-nn:9000/warehouse/projeto.db/streaming_credits/"

In [6]:
spark.sql("""
    CREATE EXTERNAL TABLE IF NOT EXISTS projeto.streaming_credits (
        person_id INT,
        id VARCHAR(50),
        name VARCHAR(80),
        character VARCHAR(300),
        platform VARCHAR(10),
        role VARCHAR(50)
    )
    USING DELTA
    PARTITIONED BY (role)
    LOCATION 'hdfs://hdfs-nn:9000/warehouse/projeto.db/streaming_credits/'
""")

DataFrame[]

In [7]:
spark.sql("""
    INSERT OVERWRITE TABLE projeto.streaming_credits
    PARTITION (role)
    
    SELECT 
        person_id, id, name, character, 
        'netflix' as platform,
        role
    FROM projeto.netflix_credits
    
    UNION ALL
    
    SELECT 
        person_id, id, name, character, 
        'amazon' as platform,
        role
    FROM projeto.amazon_credits
""")

DataFrame[]

In [8]:
spark.sql("""
    SELECT platform, COUNT(*) as total_count
    FROM projeto.streaming_credits
    GROUP BY platform
""").show()

+--------+-----------+
|platform|total_count|
+--------+-----------+
|  amazon|     124235|
| netflix|      77801|
+--------+-----------+



In [10]:
spark.sql("""
    SELECT * FROM projeto.streaming_credits
    LIMIT 20
""").show()

+---------+-------+------------------+--------------------+--------+-----+
|person_id|     id|              name|           character|platform| role|
+---------+-------+------------------+--------------------+--------+-----+
|     3748|tm84618|    Robert De Niro|       Travis Bickle| netflix|ACTOR|
|    14658|tm84618|      Jodie Foster|       Iris Steensma| netflix|ACTOR|
|     7064|tm84618|     Albert Brooks|                 Tom| netflix|ACTOR|
|     3739|tm84618|     Harvey Keitel|Matthew 'Sport' H...| netflix|ACTOR|
|    48933|tm84618|   Cybill Shepherd|               Betsy| netflix|ACTOR|
|    32267|tm84618|       Peter Boyle|              Wizard| netflix|ACTOR|
|   519612|tm84618|    Leonard Harris|Senator Charles P...| netflix|ACTOR|
|    29068|tm84618|    Diahnne Abbott|     Concession Girl| netflix|ACTOR|
|   519613|tm84618|       Gino Ardito|  Policeman at Rally| netflix|ACTOR|
|     3308|tm84618|   Martin Scorsese|Passenger Watchin...| netflix|ACTOR|
|    43791|tm84618|     M

In [11]:
spark.stop()